Note: This module is just for analyzing distances between same persons and different persons.

Right now we are using:

`threshold = 0.8`

But that number is random.

A real system NEVER guesses threshold.

### Why This Matters

Your system outputs distances like:

    0.90  (same person)
    1.18  (different person)
    1.29
    1.34

But what if:

- Another same-person image gives 1.05?

- Another different-person gives 0.92?

Now threshold becomes critical.

### Core Concept: Distance Distributions

here are only two types of comparisons:

- Same person (positive pairs)
- Different person (negative pairs)

You must understand how their distances are distributed.

Think like this:

    Same person distances:
    0.55
    0.61
    0.72
    0.83
    0.94

    Different person distances:
    1.10
    1.22
    1.30
    1.45

The gap between them is your safety zone.

In [1]:
import torch
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image
import os
import itertools

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(image_size=160, margin=0, device=device)

resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [3]:
database = {}

for person in os.listdir('faces/single'):
    for file in os.listdir(f'faces/single/{person}'):
        img = Image.open(os.path.join(f'faces/single/{person}', file))
        
        face = mtcnn(img)
        
        if face is None:
            continue
        
        face = face.unsqueeze(0).to(device)
        
        with torch.no_grad():
            embedding = resnet(face)
            
        database[file] = embedding

In [4]:
same_distances = []
different_distances = []

for (name1, emb1), (name2, emb2) in itertools.combinations(database.items(), 2):
    
    # Normalize both embeddings
    emb1 = emb1 / emb1.norm(dim=1, keepdim=True)
    emb2 = emb2 / emb2.norm(dim=1, keepdim=True)
    
    distance = torch.norm(emb1 - emb2).item()
    
    # Extract identity (remove numbers + extension)
    name1 = name1.split('.')[0]
    name2 = name2.split('.')[0]
    id1 = ''.join(filter(str.isalpha, name1))
    id2 = ''.join(filter(str.isalpha, name2))
    
    print(id1, id2)
    
    if(id1 == id2):
        same_distances.append(distance)
    else:
        different_distances.append(distance)
        
print("Same person distances:")
print(same_distances)

print("\nDifferent person distances:")
print(different_distances)

abhishekbachchan abhishekbachchan
abhishekbachchan abhishekbachchan
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan shail
abhishekbachchan shail
abhishekbachchan abhishekbachchan
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan shail
abhishekbachchan shail
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan ab
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan jayabachchan
abhishekbachchan shail
abhishekbachchan shail
ab ab
ab ab
ab ab
ab ab
ab jayabachchan
ab jayabachchan
ab jayabachchan
ab jayabachchan
ab shail
ab 

`itertools.combinations(database.items(), 2)`

This creates all unique pairs.

Example:

- (A,B)
- (A,C)
- (B,C)

No duplicates.